In [11]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
import ast
from collections import Counter


#CARGA DE DATOA
train_data = pd.read_csv("../../Data/OnlyOneEmotion/train_emotions.csv")
test_conflictive = pd.read_csv("../../Data/OnlyOneEmotion/test_emotions_conflicting.csv")
test_clean_data = pd.read_csv("../../Data/OnlyOneEmotion/test_emotions_complete.csv")

# Convertir las cadenas de listas en listas reales
all_emotions = test_conflictive['Emotion'].apply(ast.literal_eval)

# Aplanar todas las listas en una sola lista
flattened_emotions = [emotion for sublist in all_emotions for emotion in sublist]

# Contar ocurrencias de cada emoción
emotion_counts = Counter(flattened_emotions)

#Reentrenamiento del modelo con SVM después de comprobar que es el mejor modelo, (usamos el train_data original, sin división, y ambos archivos de test, uno para etiquetar y añadirlo al train y otro para predecir y evaluar su desempeño con y sin aumento de datos)

#Vectorizar los textos
vectorizer = TfidfVectorizer(max_features=10000, lowercase=True, strip_accents='unicode')

# Reentrenamiento del modelo con SVM
X_full_train = vectorizer.fit_transform(train_data['Text'])
X_test_clean = vectorizer.transform(test_clean_data['Text'])
X_test_conflictive = vectorizer.transform(test_conflictive['Text'])

y_full_train = train_data['Emotion']
y_test_clean = test_clean_data['Emotion']
y_test_conflictive = test_conflictive['Emotion']

clf = SVC(kernel='linear', class_weight='balanced')

print("Reentrenando el modelo con SVM...")
clf.fit(X_full_train, y_full_train)

#Predicción sobre el conjunto test limpio
y_pred_test_clean = clf.predict(X_test_clean)
print(f"Predicciones en el conjunto test limpio:\n {classification_report(y_test_clean, y_pred_test_clean, zero_division=0)}")




Reentrenando el modelo con SVM...
Predicciones en el conjunto test limpio:
               precision    recall  f1-score   support

           0       0.63      0.55      0.59       348
           1       0.75      0.80      0.77       186
           2       0.39      0.47      0.42       131
           3       0.23      0.22      0.22       194
           4       0.25      0.29      0.27       236
           5       0.16      0.35      0.22        86
           6       0.21      0.37      0.27        97
           7       0.25      0.39      0.31       176
           8       0.28      0.43      0.34        56
           9       0.18      0.27      0.21        88
          10       0.25      0.40      0.31       195
          11       0.42      0.50      0.46        76
          12       0.25      0.26      0.26        23
          13       0.25      0.51      0.33        57
          14       0.59      0.65      0.62        65
          15       0.93      0.87      0.90       260
     

In [16]:


# Predicción sobre el conjunto test conflictivo
y_pred_test_conflictive = clf.predict(X_test_conflictive)

# Crear DataFrame con las predicciones del test conflictivo
new_test_conflictive_data = pd.DataFrame({
    'Text': test_conflictive['Text'],
    'Emotion': y_pred_test_conflictive
})

#Crear otro DatraFrame para inspeccion manual de las predicciones del test conflictivo
comparation_df = pd.DataFrame({
    'Text': test_conflictive['Text'],
    'Original Emotion': test_conflictive['Emotion'],
    'Predicted Emotion': y_pred_test_conflictive
})

print(f"Predicciones del test conflictivo:\n{new_test_conflictive_data.head()}")

print("\nCuenta de emociones en la predicción:\n")
print(comparation_df["Predicted Emotion"].value_counts())

# Guardar las predicciones del test conflictivo en un archivo CSV
comparation_df.to_csv("../../Data/OnlyOneEmotion/ToCompareResults.csv", index=False)

#Concatenar el DataFrame de test conflictivo con el train original para volver a entrenar el modelo
new_train_data_conflictive = pd.concat([train_data, new_test_conflictive_data], ignore_index=True)


# Asegurar que la columna Original Emotion esté en formato lista
comparation_df['Original Emotion'] = comparation_df['Original Emotion'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# Filtrar solo los casos donde la predicción esté contenida en la lista de emociones originales
correct_predictions_df = comparation_df[
    comparation_df.apply(lambda row: row['Predicted Emotion'] in row['Original Emotion'], axis=1)
].copy()

print("\nFilas donde la predicción coincide con al menos una emoción original:")
print(correct_predictions_df.head())

print(f"\nNúmero total de aciertos parciales: {len(correct_predictions_df)} / {len(comparation_df)}")


correct_predictions_df.to_csv("../../Data/OnlyOneEmotion/PrediccionesCorrectas.csv", index=False)

#Class weight balanced --> No funcionó

#Hacer una limpieza del etiquetado en bruto de aquellos que si predijo de forma correcta.

Predicciones del test conflictivo:
                                                Text  Emotion
0  We need more boards and to create a bit more s...       11
1  Aww... she'll probably come around eventually,...       27
2  Shit, I guess I accidentally bought a Pay-Per-...       27
3  Maybe that’s what happened to the great white ...       27
4  I never thought it was at the same moment, but...       27

Cuenta de emociones en la predicción:

Predicted Emotion
27    6168
11     281
0      223
10     154
5      122
3      117
7       95
14      85
1       76
13      74
17      71
20      69
25      68
2       65
22      61
26      42
4       34
9       31
12      16
18      16
8       16
19      15
6       15
23       9
24       8
15       5
16       3
Name: count, dtype: int64

Filas donde la predicción coincide con al menos una emoción original:
                                                 Text Original Emotion  \
4   I never thought it was at the same moment, but...       [6, 9, 

In [13]:
#Segundo reentrenamiento del modelo con SVM usando el train con prediccions limpias
print(correct_predictions_df.head())


print("Valores por emoción:")
print(correct_predictions_df['Predicted Emotion'].value_counts())

correct_predictions_df['Emotion'] = correct_predictions_df['Predicted Emotion']

toConcat_df = pd.DataFrame({
    'Text': correct_predictions_df['Text'],
    'Emotion': correct_predictions_df['Emotion'],
})


print(toConcat_df.head())

new_train_data_correct = pd.concat([train_data, toConcat_df], ignore_index=True)

X_new_train_correct = vectorizer.fit_transform(new_train_data_correct['Text'])
y_new_train_correct = new_train_data_correct['Emotion']

clf = SVC(kernel='linear', class_weight='balanced')

print("Reentrenando el modelo con SVM con el nuevo conjunto de entrenamiento (limpio)...")
clf.fit(X_new_train_correct, y_new_train_correct)

# Predicción sobre el conjunto test limpio con el nuevo modelo
y_pred_new_test_clean_correct = clf.predict(X_test_clean)
print(f"Predicciones en el conjunto test limpio con el nuevo modelo:\n {classification_report(y_test_clean, y_pred_new_test_clean_correct, zero_division=0)}")




                                                Text Original Emotion  \
0  We need more boards and to create a bit more s...          [8, 20]   
1  Aww... she'll probably come around eventually,...           [1, 4]   
5                            I miss them being alive         [16, 25]   
6        Ok, then what the actual fuck is your plan?           [2, 7]   
7                    aw, thanks! I appreciate that!           [0, 15]   

   Predicted Emotion  
0                 20  
1                  1  
5                 25  
6                  2  
7                 15  
Valores por emoción:
Predicted Emotion
15    684
18    494
1     438
0     430
27    303
20    213
17    212
7     208
4     204
2     186
3     180
10    171
26    144
25    143
5     134
6     132
24    121
11     95
8      90
22     84
13     83
9      82
14     66
12     30
21     18
19     12
16     10
23      2
Name: count, dtype: int64
                                                Text  Emotion
0  We need more 

In [14]:
#Tercer reentrenamiento del modelo con SVM usando el nuevo train que incluye las predicciones del test conflictivo
X_new_train_conflictive = vectorizer.fit_transform(new_train_data_conflictive['Text'])
y_new_train_confilctive = new_train_data_conflictive['Emotion']

clf = SVC(kernel='linear', class_weight='balanced')

print("Reentrenando el modelo con SVM con el nuevo conjunto de entrenamiento (conflictivo)...")
clf.fit(X_new_train_conflictive, y_new_train_confilctive)

# Predicción sobre el conjunto test limpio con el nuevo modelo
y_pred_new_test_clean_conflictive = clf.predict(X_test_clean)
print(f"Predicciones en el conjunto test conflictivo con el nuevo modelo: {classification_report(y_test_clean, y_pred_new_test_clean_conflictive, zero_division=0)}")

Reentrenando el modelo con SVM con el nuevo conjunto de entrenamiento (conflictivo)...
Predicciones en el conjunto test conflictivo con el nuevo modelo:               precision    recall  f1-score   support

           0       0.01      0.00      0.00       348
           1       0.04      0.01      0.02       186
           2       0.04      0.02      0.02       131
           3       0.01      0.01      0.01       194
           4       0.00      0.00      0.00       236
           5       0.03      0.02      0.02        86
           6       0.00      0.00      0.00        97
           7       0.06      0.02      0.03       176
           8       0.00      0.00      0.00        56
           9       0.07      0.01      0.02        88
          10       0.02      0.01      0.01       195
          11       0.01      0.01      0.01        76
          12       0.00      0.00      0.00        23
          13       0.00      0.00      0.00        57
          14       0.12      0.08   

In [15]:

# --- DIRECTORIO DE SALIDA DE LOS PLOTS ---
output_dir = "../../Plots/Experiment2/"


import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Función para graficar matriz de confusión
def plot_confusion_matrix(y_true, y_pred, labels, title):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(labels))))

    plt.figure(figsize=(12, 10))
    ax = sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues", 
        xticklabels=labels,
        yticklabels=labels,
        cbar=True,
        annot_kws={"size": 6}
    )
    ax.set_title(f'Matriz de Confusión - {title}', fontsize=12)
    ax.set_xlabel('Etiqueta Predicha', fontsize=10)
    ax.set_ylabel('Etiqueta Verdadera', fontsize=10)

    # Reducir tamaño de etiquetas en los ejes
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=6)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=6)
    
    plt.tight_layout()
    # Guardar archivo
    filename = title.replace(" ", "_").replace("(", "").replace(")", "").lower() + ".png"
    plt.savefig(output_dir + filename)
    plt.close()



emotion_labels = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity',
    'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear',
    'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief',
    'remorse', 'sadness', 'surprise', 'neutral'
]
#Sin aumento de datos
plot_confusion_matrix(
    y_true=y_test_clean,
    y_pred=y_pred_test_clean,
    labels=emotion_labels,
    title="Original sin aumento de datos"
)

#Con aumento limpio
plot_confusion_matrix(
    y_true=y_test_clean,
    y_pred=y_pred_new_test_clean_correct,
    labels=emotion_labels,
    title="Con aumento de datos (predicciones correctas)"
)

#Con aumento conflictivo
plot_confusion_matrix(
    y_true=y_test_clean,
    y_pred=y_pred_new_test_clean_conflictive,
    labels=emotion_labels,
    title="Con aumento de datos (predicciones conflictivas)"
)



    
    